# 06 · Анализ по неделям (ISO)

**Цель** — динамика событий и тревог на уровне недель, а не годов: сезонность, всплески, стабильность классов. Недельный горизонт ближе к реальному мониторингу (диспетчер смотрит риски на 24ч–7д) и позволяет увидеть короткие аномалии, которые «съедаются» годовыми агрегатами.

In [ ]:
import sys
import pathlib

_HERE = pathlib.Path.cwd()
if _HERE.name == "notebooks":
    RESEARCH = _HERE.parent
else:
    RESEARCH = _HERE
if str(RESEARCH) not in sys.path:
    sys.path.insert(0, str(RESEARCH))

import pandas as pd
import numpy as np
import data_utils as du

# True — полный пересчёт из сырых CSV (медленно, ~10-15 мин).
# False — читать готовый артефакт weekly_summary.csv (быстро).
RECOMPUTE = False

print("dataset:", du.find_dataset_dir())

# 6.1 · Недельная сводка (артефакт `weekly_summary.csv`)

Для каждого события определяем ISO-неделю (`год-неделя`). Считаем по неделе:
число событий, число тревог, уникальных каналов, долю тревог. Отдельно — тревоги по неделям для ключевых типов датчиков.

In [ ]:
out_csv = du.find_dataset_dir() / "weekly_summary.csv"
out_al = du.find_dataset_dir() / "weekly_alarms_by_type.csv"

if RECOMPUTE or not out_csv.exists():
    ref_ch = du.load_ref_channels()[["ид_канала_данных", "тип_датчика"]]

    # Аккумуляторы: простые pandas-агрегаты по чанкам
    acc = []          # (год_неделя, событий, тревог, каналов)
    acc_al = []       # (год_неделя, тип_датчика, тревог)

    for fp in du.journal_files():
        year = fp.name.split("-")[2].split(".")[0]
        for ch in du.read_chunks(fp, cols=["дата", "тревожное", "ид_канала_данных"]):
            dt = pd.to_datetime(ch["дата"])
            iso = dt.dt.isocalendar()
            wk = iso["year"].astype("int64") * 100 + iso["week"].astype("int64")

            g = (ch.assign(wk=wk, alarm=ch["тревожное"].eq("t"))
                   .groupby("wk", sort=False))
            s = g["alarm"].size().rename("событий")
            a = g["alarm"].sum().astype("int64").rename("тревог")
            c = g["ид_канала_данных"].nunique().rename("каналов")
            acc.append(pd.concat([s, a, c], axis=1).reset_index())

            # тревоги по типам датчиков
            al = ch[ch["тревожное"].eq("t")]
            if len(al):
                typ = al["ид_канала_данных"].map(dict(zip(ref_ch["ид_канала_данных"],
                                                          ref_ch["тип_датчика"]))
                                                 ).fillna("(нет в справочнике)")
                tmp = pd.DataFrame({
                    "wk": wk.loc[al.index],
                    "тип_датчика": typ,
                    "alarm": 1,
                })
                g2 = tmp.groupby(["wk", "тип_датчика"], sort=False)["alarm"].sum().rename("тревог")
                acc_al.append(g2.reset_index())
        print(f"[{year}] done, недель в акк: {sum(x['wk'].nunique() for x in acc)}", flush=True)

    weekly = pd.concat(acc).groupby("wk", as_index=False).sum(numeric_only=True)
    weekly.columns = ["год_неделя", "событий", "тревог", "каналов"]
    weekly["год"] = (weekly["год_неделя"] // 100).astype(int)
    weekly["неделя"] = (weekly["год_неделя"] % 100).astype(int)
    weekly["доля_тревог_%"] = (100 * weekly["тревог"] / weekly["событий"]).round(4)
    weekly = weekly.sort_values("год_неделя").reset_index(drop=True)
    weekly.to_csv(out_csv, index=False, encoding="utf-8-sig")

    at = pd.concat(acc_al).groupby(["wk", "тип_датчика"], as_index=False)["тревог"].sum()
    at = at.rename(columns={"wk": "год_неделя"})
    at.to_csv(out_al, index=False, encoding="utf-8-sig")
else:
    weekly = pd.read_csv(out_csv, dtype={"год_неделя": str, "год": str, "неделя": str})
    at = pd.read_csv(out_al, dtype={"год_неделя": str})
    print("Прочитано из готовых артефактов")

print("Недель всего:", len(weekly))
weekly.head(10)

# 6.2 · Всплески и сезонность

Смотрим на самые аномальные недели (по тревогам) и рисуем тренд. Нас интересует: какие недели выбиваются, есть ли сезонная компонента (зима/лето) и как «дышит» класс тревог.

In [ ]:
import matplotlib.pyplot as plt

# Топ-15 недель по числу тревог
top_w = weekly.nlargest(15, "тревог")[["год_неделя", "событий", "тревог", "доля_тревог_%"]]
print("Топ-15 недель по тревогам:")
print(top_w.to_string(index=False))

# Среднее по неделям года (сезонная компонента)
season = weekly.groupby("неделя")[["событий", "тревог", "доля_тревог_%"]].mean()
print("\nСредне-недельная картина по номеру недели (1..52):")
print(season.reset_index().sort_values("тревог", ascending=False).head(10).to_string(index=False))

# 6.3 · Графики: динамика по неделям

Три панели: события, тревоги, доля тревог — по непрерывной оси (год_неделя). Отдельными линиями — ключевые типы датчиков (тревоги по неделям).

In [ ]:
import matplotlib.pyplot as plt

w = weekly.copy()
w["x"] = w["год_неделя"].astype(int)

fig, axes = plt.subplots(3, 1, figsize=(15, 11), sharex=True)
w.plot(x="x", y="событий", ax=axes[0], legend=False, title="Событий по неделям")
axes[0].set_xlabel("")
w.plot(x="x", y="тревог", ax=axes[1], color="C3", legend=False, title="Тревог по неделям")
axes[1].set_xlabel("")
w.plot(x="x", y="доля_тревог_%", ax=axes[2], color="C2", legend=False, title="Доля тревог %")
axes[2].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

# Тревоги ключевых типов по неделям
key = ["Датчик дыма", "Датчик движения", "Состояние насоса", "Газовый датчик"]
atk = at[at["тип_датчика"].isin(key)].copy()
atk["x"] = atk["год_неделя"].astype(int)
piv = atk.pivot_table(index="x", columns="тип_датчика", values="тревог", aggfunc="sum", fill_value=0)
piv.plot(figsize=(15, 5), title="Тревоги ключевых типов по неделям")
plt.tight_layout()
plt.show()

# 6.4 · Комментарии

- **Главная недельная аномалия — 2021, недели 17–22 (апрель–май)**: до **241 тыс. тревог/неделю** (доля тревог до 17.7%!). По типам — это **990 тыс. тревог датчиков дыма** за 6 недель (**~85% всех тревог дыма за 8 лет**!). Параллельно в 2021 у дыма было **1.17 млн статусов «Неисправен»** — это не пожары, а **массовая проверка/ремонт извещателей**.
- **Регуляторный контекст (проверено по официальным текстам)**:
  - **ППР №1479** («Правила противопожарного режима») вступили в силу **01.01.2021** (п.2 постановления, docs.cntd.ru) — новые требования к содержанию систем АПС.
  - **С 01.03.2021** вступили в силу новые своды правил МЧС по пожарной сигнализации — **СП 484.1311500.2020** (адресно-аналоговая сигнализация), **СП 485** (оповещение), **СП 486** (автопожаротушение).
  - Плюс постановление Правительства №2463 от 31.12.2020 (переходные положения к ППР-1479).
  - **Вывод:** пик 2021W17–W22 — это **эффект регуляторики** (кампания проверок после новых правил), а не физическая деградация парка датчиков.
- **Свежие недели 2026 (16–23)**: тревоги 2.6–14 тыс. в неделю, всплеск на неделе 23 (27 тыс.) — снова в основном дым (проверить в `weekly_alarms_by_type.csv`); природа требует отдельной проверки, помечена в календаре кампаний как подозрительная.
- **Нейтральные недели «шума»**: недели 202545, 202537 — ~1.3 млн событий при <3 тыс. тревог; здесь телеметрия гасит сигналы — нужно нормировать на канал.
- **Вывод для ML**: недели кампаний (см. `data_utils.CAMPAIGN_WEEKS`) **исключаем из сплитов** train/val/test, иначе модель выучит «эффект проверки» вместо поведения датчиков. Недельные окна + нормировка по каналу — обязательны.

# 6.5 · Кампании проверок: пометка и сравнение

Помечаем недели кампаний (`data_utils.CAMPAIGN_WEEKS`) в сводке и сравниваем
«чистые» недели с аномальными (2021W17–W22). Это основа правила
«исключать кампании из сплитов» перед обучением.

In [ ]:
# Помечаем кампании
w2 = du.add_campaign_flags(weekly)
print("Недель кампаний:", int(w2["is_campaign"].sum()))
print(w2[w2["is_campaign"]].to_string(index=False))

# Сравнение
print("\n=== Средняя неделя: кампании vs чистые ===")
comp = w2.groupby("is_campaign")[["событий", "тревог", "доля_тревог_%"]].mean()
comp.index = ["чистые" if not i else "кампании" for i in comp.index]
print(comp.to_string())

# Чистые недели (для будущих сплитов)
clean = du.filter_clean_weeks(weekly)
print("\nЧистых недель остаётся:", len(clean), "из", len(weekly))